<a href="https://colab.research.google.com/github/priyadarshini-GVPW/XAI_ML/blob/main/Analysis_Drebin.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==========================================
# ONE-CLASS MALWARE MODEL (FINAL VERSION)
# ==========================================

import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import joblib

# ------------------------------------------
# 1. LOAD MALWARE DATASET (TRAINING DATA)
# ------------------------------------------
url='https://raw.githubusercontent.com/priyadarshini-GVPW/XAI_ML/main/all_malware_permissions.csv'
data = pd.read_csv(url)

print("Original Shape:", data.shape)

# Remove label column if present
if "label" in data.columns:
    data = data.drop(columns=["label"])

print("Shape after removing label:", data.shape)

# ------------------------------------------
# 2. SAVE FEATURE COLUMN NAMES (IMPORTANT)
# ------------------------------------------
feature_columns = data.columns.tolist()
joblib.dump(feature_columns, "training_feature_columns.pkl")

print("Feature columns saved.")

# ------------------------------------------
# 3. FEATURE SCALING
# ------------------------------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(data)

# Save scaler
joblib.dump(scaler, "scaler.pkl")
print("Scaler saved.")

# ------------------------------------------
# 4. TRAIN ONE-CLASS MODEL
# ------------------------------------------
model = IsolationForest(
    n_estimators=300,
    contamination=0.05,  # 5% anomaly tolerance
    random_state=42,
    n_jobs=-1
)

model.fit(X_scaled)

# Save model
joblib.dump(model, "malware_oneclass_model.pkl")

print("Model trained and saved successfully.")

# ==========================================
# OPTIONAL: TESTING ON SAME DATA (SANITY CHECK)
# ==========================================

predictions = model.predict(X_scaled)

# Convert predictions to readable form
readable_predictions = np.where(predictions == 1,
                                "Malware-like",
                                "Not-Malware-like")

print("\nPrediction Summary:")
print(pd.Series(readable_predictions).value_counts())

# Save training prediction results
output = data.copy()
output["Prediction"] = readable_predictions
output.to_csv("training_predictions_output.csv", index=False)

print("\nTraining predictions saved.")

Original Shape: (5481, 285)
Shape after removing label: (5481, 285)
Feature columns saved.
Scaler saved.
Model trained and saved successfully.

Prediction Summary:
Malware-like        5207
Not-Malware-like     274
Name: count, dtype: int64

Training predictions saved.


In [9]:
# ================================================
# APK MALWARE DETECTION PIPELINE
# ================================================

!pip install androguard

import pandas as pd
import numpy as np
import joblib
from androguard.misc import AnalyzeAPK
from sklearn.preprocessing import StandardScaler

# -----------------------------------------------
# 1. Load trained model, scaler, feature columns
# -----------------------------------------------
model = joblib.load("malware_oneclass_model.pkl")
scaler = joblib.load("scaler.pkl")
feature_columns = joblib.load("training_feature_columns.pkl")

print("Model, scaler and feature list loaded successfully.")

# -----------------------------------------------
# 2. Function to extract permissions from APK
# -----------------------------------------------
def extract_permissions(apk_path):
    a, d, dx = AnalyzeAPK(apk_path)
    permissions = a.get_permissions()
    return permissions

# -----------------------------------------------
# 3. Convert permissions to feature vector
# -----------------------------------------------
def create_feature_vector(permissions, feature_columns):

    # Initialize all features to 0
    feature_dict = {feature: 0 for feature in feature_columns}

    # Set 1 if permission exists
    for perm in permissions:
        if perm in feature_dict:
            feature_dict[perm] = 1

    return pd.DataFrame([feature_dict])

# -----------------------------------------------
# 4. Upload and test APK
# -----------------------------------------------
!wget -O app.apk https://raw.githubusercontent.com/priyadarshini-GVPW/XAI_ML/main/00c8de6b31090c32b65f8c30d7227488d2bce5353b31bedf5461419ff463072d
apk_path = "app.apk"
permissions = extract_permissions(apk_path)

print("Extracted Permissions:", permissions)

apk_features = create_feature_vector(permissions, feature_columns)

# -----------------------------------------------
# 5. Scale features
# -----------------------------------------------
apk_scaled = scaler.transform(apk_features)

# -----------------------------------------------
# 6. Predict
# -----------------------------------------------
prediction = model.predict(apk_scaled)
score = model.decision_function(apk_scaled)

# -----------------------------------------------
# 7. Interpret result
# -----------------------------------------------
if prediction[0] == 1:
    result = "Malware-like"
else:
    result = "Not Malware-like (Possibly Benign)"

print("\n==============================")
print("Prediction:", result)
print("Anomaly Score:", score[0])
print("==============================")

Model, scaler and feature list loaded successfully.
--2026-03-04 05:20:24--  https://raw.githubusercontent.com/priyadarshini-GVPW/XAI_ML/main/00c8de6b31090c32b65f8c30d7227488d2bce5353b31bedf5461419ff463072d
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 110644 (108K) [application/octet-stream]
Saving to: ‘app.apk’

app.apk             100%[===================>] 108.05K  --.-KB/s    in 0.002s  

2026-03-04 05:20:24 (50.4 MB/s) - ‘app.apk’ saved [110644/110644]



2026-03-04 05:20:24.459 | DEBUG    | androguard.misc:AnalyzeAPK:50 - AnalyzeAPK
2026-03-04 05:20:24.460 | DEBUG    | androguard.misc:AnalyzeAPK:65 - Analysing without session
2026-03-04 05:20:24.464 | INFO     | androguard.core.apk:_apk_analysis:410 - Starting analysis on AndroidManifest.xml
2026-03-04 05:20:24.466 | DEBUG    | androguard.core.axml:__init__:1129 - AXMLPrinter
2026-03-04 05:20:24.467 | DEBUG    | androguard.core.axml:__init__:449 - AXMLParser
2026-03-04 05:20:24.468 | DEBUG    | androguard.core.axml:__init__:482 - FIRST HEADER <ARSCHeader idx='0x00000000' type='3' header_size='8' size='5932'>
2026-03-04 05:20:24.469 | DEBUG    | androguard.core.axml:__init__:540 - STRING_POOL <ARSCHeader idx='0x00000008' type='1' header_size='28' size='2556'>
2026-03-04 05:20:24.470 | DEBUG    | androguard.core.axml:is_valid:575 - True
2026-03-04 05:20:24.472 | DEBUG    | androguard.core.axml:_do_next:593 - M_EVENT -1
2026-03-04 05:20:24.473 | DEBUG    | androguard.core.axml:_do_next:60

Extracted Permissions: ['com.google.android.c2dm.permission.RECEIVE', 'android.permission.READ_SMS', 'android.permission.WRITE_EXTERNAL_STORAGE', 'com.software.application.permission.C2D_MESSAGE', 'android.permission.INTERNET', 'android.permission.SEND_SMS', 'android.permission.READ_PHONE_STATE', 'android.permission.RECEIVE_SMS', 'android.permission.WAKE_LOCK']

Prediction: Malware-like
Anomaly Score: 0.1354570650473771


In [10]:
# 7. Interpret result
# -----------------------------------------------
if prediction[0] == 1:
    result = "Malware-like"
else:
    result = "Not Malware-like (Possibly Benign)"

print("\n==============================")
print("Prediction:", result)
print("Anomaly Score:", score[0])
print("==============================")


Prediction: Malware-like
Anomaly Score: 0.1354570650473771
